Define Results Tag
---

- introduced on 2026 -07 -14, to avoid overlapping results for same configuration.
- RUNTAG is handled automatically when scenario_pipeline is run. this notebook shows a demo on how this RUNTAG is being used to define paths.

In [ ]:
from datetime import datetime
RUNTAG = datetime.now().strftime("%Y%m%d")

# <span style="color: Coral;"> Scenario config UI </span>

- Load the <span style="color: orange;"> _utils_ </span> module, which handles the backend functions to provide the <span style="color: blue;"> __scenario comnfig ui__ </span> 

In [ ]:
import bcnexus.utils as utils
import re # For regex operations
scenarios_dd, storage_algo_dd, h_grouping_dd, n_clusters_dd, timeslices_label = utils.build_scenario_ui()

- Assign the <span style="color: orange;"> __values__</span>  from <span style="color: blue;"> __scenario UI dropdowns__</span> to <span style="color: orange;"> __arguments__</span> 

In [ ]:
model_builder_args = dict(
    run_scenario=scenarios_dd.value ,
    storage_algorithm=storage_algo_dd.value,
    clustering_attributes=dict(
        hour_grouping=h_grouping_dd.value,
        n_clusters=n_clusters_dd.value
    )
)


# We use the Timeslices information to access run specific folders
timeslices_int = int(re.search(r'\d+', timeslices_label.value).group()) if re.search(r'\d+', timeslices_label.value) else None


# <span style="color: Coral;"> Model Runner Object </span>

* <span style="color: grey;">From `clews.runner` module, load the `RunModel` Object </span>  

In [ ]:
from bcnexus.clews.runner import RunModel

* <span style="color: grey;"> Create an instance of the `RunModel` Object `bcnexusRun` with __model_builder_args__ </span>  

In [ ]:
bcnexusRun = RunModel(**model_builder_args)

### 🚀 `run()` — Core Execution Workflow

* <span style="color: grey;"> Apply the `run()` method from  `bcnexusRun` instance </span> to run the complete workflow

About __Arguments__ of the `run()` method:

- `build=True` argument handles the workflow run to build SETs, Ratios, template params etc.
- `include_livestock` handles the livestock modelling workflow using `bcnexus/clews/livestock.py` module.
- `thread` depends on the hardware limitations of your machine. If you have 4 core CPU, use Thread __<=4__ )

* <span style="color: skyblue;"> Where do I get `threads` number ?</span>  
  - <span style="color: grey;"> Apply the `check_machine_cores()` function from  `utils` module. it returns the __number of cores available__ </span>  

In [ ]:
n_physical_cores, n_logical_cores=utils.check_machine_cores()

| Model Characteristics                         | Recommended Threads | Reasoning                                                      |
| --------------------------------------------- | ------------------- | -------------------------------------------------------------- |
| Small LP/MILP (<50k vars)                     | 1–4                 | Over-parallelization adds overhead; single-thread often enough |
| Medium MILP (50k–500k vars)                   | 4–8                 | Good balance of parallel speed-up and stability                |
| Large/complex MILP (>500k vars, branch-heavy) | 8–16                | Parallel branch-and-bound scales moderately well               |
| Nonlinear / Quadratic / MIQP                  | 4–8                 | Solvers often rely on sequential matrix operations             |
| Highly degenerate or network-based LP         | 1–4                 | Parallel simplex often unstable; dual simplex preferred        |


* <span style="color: orange;">  Now run the Model ! </span>  

 <span style="color: coral;"> Important Notes:
  - While creating an instance of `RunModel` object (here named as `bcnexusRun`), it also creates an instance of `BuildModel` object (initiated and named as `ClewsBuilder` inside `RunModel` object ). You can access it via `bcnexusRun.ClewsBuilder`  , and access the methods inside `ClewsBuilder` object. 
  - The `ClewsBuilder` object can be found at </span> <span style="color: biege;"> bcnexus/clews/__builder.py__ module
  - when we set `build=True`. We use the `build()` method of `ClewsBuilder` object.
    - The `build()` method is useful to test/validate new structural changes to the model. This method handles the SETs creation, collection of LandCluster data, technology updates via clews_builder.yaml config. Refer to the script and the example workbook for more details on this object and methods.
    - `ClewsBuilder` has an explicit method `build_SETs_and_ratios` which is an adaptation of the former __clewsy__ tool. Developers can use this method and add their custom methods to incorporate new structural changes to the model.

<span style="color: coral;"> Data flow in folder:</span>
  - Everything happens inside : <span style="color: coral;"> data/clews_data</span>
  - The workflow __creates this directory__ : <span style="color: coral;">data/clews_data/<span style="color: magenta;">clews_build_data </span><br>
      <span style="color: grey;"> It has 3 sub-folders: 
      <span style="color: coral;">
      - input_csvs  <span style="color: grey;">: The workflow collects templates data first, then updates the files accordingly to reflect changes in clews_builder.yaml config. <span style="color: red;"> if `build`=False, then the workflow assumes all files are updated in this folder. Once you build the model structure, to adjust the temporal clustering attributes only, you can set `build=False`, to skip the steps to build this folder.
      - inputs_csv_8760<span style="color: grey;">: Creates BCNexus compatible profiling parameters in hourly resolution. Which are used to apply the clustering. <span style="color: red;"> if `build`=False, then the workflow assumes all files are updated in this folder.Once you build the model structure, to adjust the temporal clustering attributes only, you can set `build=False`, to skip the steps to build this folder.
      - Model_storage_algorithm<span style="color: grey;">: Final datafiles are stored inside this folder with storage case, and sub-folders with scenarios. 
  - Once you build the model structure, yt adjust the clustering attributes only, you can set `build=False`, to skip the steps to build this folder.


In [ ]:
bcnexusRun.run(build=True,
             include_livestock=False,
             threads=16)

DATA_INFILE: data/clews_data/clews_build_data/Model_Kotzur/Base_CNZ/Base_CNZ_data.txt IS_DIR? False
 └> Preprocessing data and model file for case: Kotzur
  └> data file processed and saved as : data/clews_data/clews_build_data/Model_Kotzur/Base_CNZ/Base_CNZ_dataProcessed.txt
  └> model file processed and saved as : data/clews_data/clews_build_data/Model_Kotzur/Base_CNZ/Kotzur_Model_processed.txt
└> Preparing the LP file
 └> Creating LP file: data/clews_data/clews_build_data/Model_Kotzur/Base_CNZ/Base_CNZ.lp from Model : data/clews_data/clews_build_data/Model_Kotzur/Base_CNZ/Kotzur_Model_processed.txt, Data : data/clews_data/clews_build_data/Model_Kotzur/Base_CNZ/Base_CNZ_dataProcessed.txt
 ─> GLPSOL: GLPK LP/MIP Solver, v4.65
 ─> Parameter(s) specified in the command line:
 ─> -m data/clews_data/clews_build_data/Model_Kotzur/Base_CNZ/Kotzur_Model_processed.txt
 ─> -d data/clews_data/clews_build_data/Model_Kotzur/Base_CNZ/Base_CNZ_dataProcessed.txt
 ─> --wlp data/clews_data/clews_build

### Inspect Infeasible solutions (if found)

In [ ]:
# import bcnexus.clews.solver as solver
# solver.get_infeasibility_report(
#                                 lp_path=f'data/clews_data/clews_build_data/Model_Kotzur/{model_builder_args["run_scenario"]}/{model_builder_args["run_scenario"]}.lp', 
#                                 save_to=f'results/clews/Model_Kotzur_{model_builder_args["run_scenario"]}/{bcnexusRun.timeslices}ts/{bcnexusRun.timeslices}ts_{RUNTAG}_infeasible.ilp')

# <span style="color: Coral;"> Results </span>

### <span style="color: grey;"> Check Results @ `results\clews\<Model_<storage_algorithm>_<scenario>`</span>

## Get RESULTS in csvs

In [ ]:
nexus_results_root=f'results/clews/Model_{model_builder_args['storage_algorithm']}_{model_builder_args['run_scenario']}/{timeslices_int}ts/{timeslices_int}ts_csvs_gurobi_{RUNTAG}'

results_path=bcnexusRun.get_result_csvs(debug_mode=True,
                                        results_save_to=nexus_results_root)

- <span style="color: grey;"> We can also use the `datapackage` module's `GetDataPackage` object to load load the results as an object.
- <span style="color: grey;"> This results pack could be used for __post-analysis, result exchange to other models, visualizations, scenario dashes__ etc.

- <span style="color: grey;">  Get `result_pack` as an instance of `GetDataPackage` object

In [ ]:
from bcnexus.clews.datapackage import GetDataPackage
result_pack=GetDataPackage(nexus_results_root)
result_pack.show

- <span style="color: grey;">  Apply `get_dataframe(<Result param name>)` method of the `result_pack` instance

In [ ]:
#example
result_pack.get_dataframe('CapitalInvestment')

# <span style="color: yellow;"> Plots </span>

!! <span style="color: biege;"> Still under active development to include more comprehensive plots <span style="color: yellow;"> EL_20251119

- <span style="color: biege;">  Load the `plotter` module. 
  - <span style="color: grey;"> plotter.`get_plots()` function returns a dictionary of plots. You can review the plots in this notebook from that dictionary. The function also save the plots as html to 'vis/bccm/< scenario > '

- Complete workflow to create and save plots to defined path

In [ ]:
from pathlib import Path
import bcnexus.plots as plotter

plotter.main(nexus_scenario=model_builder_args['run_scenario'],
             storage_algorithm=model_builder_args['storage_algorithm'],
             timeslices=bcnexusRun.timeslices,
             results_csvs=results_path,
             plots_save_to=Path(f'vis/{model_builder_args["run_scenario"]}/{bcnexusRun.timeslices}ts_{RUNTAG}'),)

- Loading _nexus_plots_ as a dictionary of plot

In [ ]:

plotter_args=dict(
    nexus_scenario=model_builder_args['run_scenario'],
    storage_algorithm=model_builder_args['storage_algorithm'],
    timeslices=timeslices_int,
    results_csvs=nexus_results_root,
    plots_save_to=Path(f'vis/{model_builder_args["run_scenario"]}/{timeslices_int}ts_{RUNTAG}')
)
# plotter.main(**plotter_args)
nexus_plots=plotter.get_plots(**plotter_args)

- example on how to see plots inside _nexus_plots_ dict

In [ ]:
nexus_plots['Energy'].keys()

In [ ]:
nexus_plots['Energy']['power_generation_annual']

In [ ]:
nexus_plots['Energy']['power_generation_timeslices'] 

# <span style="color: Coral;"> Playground for Solved Model </span>

* <span style="color: grey;">Load the Gurobi Model object as `m` from `clewsRun`'s attribute `solved_model` </span>  

In [ ]:
m=bcnexusRun.solved_model

In [ ]:
m.printStats()          # Print basic stats: number of vars, constraints, etc.

In [ ]:
m.getAttr('Obj')     # Get objective coefficients

In [ ]:
m.getConstrs()          # List of all constraint objects

#### <span style="color: orange;">  Define or Modify the Model </span> 

In [ ]:
# x = m.addVar(lb=0, ub=10, obj=3, name='x')
# y = m.addVar(lb=0, ub=GRB.INFINITY, obj=1, name='y')
# m.addConstr(2*x + 3*y >= 10, name='demand_constraint')
# m.setObjective(3*x + 1*y, GRB.MINIMIZE)


* <span style="color: grey;">Load the `plotter` module and use the `main()` function </span>